# Octopus CUDA Strong-Strong Test

This notebook validates the CUDA path for `GaussianPoissonSolver` strong-strong collisions.

It checks CUDA visibility, builds identical CPU and GPU initial beams, runs one strong-strong collision for each public slicing method, and compares final coordinates and luminosity. The CPU and GPU paths may differ slightly because reductions and special-function approximations are not bitwise identical.

In [1]:
import Pkg
Pkg.activate(joinpath(@__DIR__, ".."))

# Load CUDA before including Octopus so precompile/load errors are visible in this cell.
cuda_load_error = nothing
cuda_load_backtrace = nothing
try
    @eval import CUDA
catch err
    global cuda_load_error = err
    global cuda_load_backtrace = catch_backtrace()
end

if cuda_load_error !== nothing
    println("CUDA failed to load in this kernel:")
    showerror(stdout, cuda_load_error, cuda_load_backtrace)
    println()
    cuda_ok = false
else
    println("CUDA loaded.")
    try
        CUDA.versioninfo()
    catch err
        println("CUDA.versioninfo() failed:")
        showerror(stdout, err, catch_backtrace())
        println()
    end
    cuda_ok = CUDA.functional(false) && CUDA.has_cuda_gpu()
    println("CUDA functional = ", CUDA.functional(false))
    println("CUDA has GPU    = ", CUDA.has_cuda_gpu())
end

octopus_load_error = nothing
octopus_load_backtrace = nothing
try
    include(joinpath(@__DIR__, "..", "src", "Octopus.jl"))
    @eval using .Octopus
catch err
    global octopus_load_error = err
    global octopus_load_backtrace = catch_backtrace()
end

if octopus_load_error !== nothing
    println("Octopus failed to load in this kernel:")
    showerror(stdout, octopus_load_error, octopus_load_backtrace)
    println()
else
    println("Octopus loaded.")
end

using Random

  Activating 

project at `/cfs/ad/dxu/Library/Julia/Octopus`


CUDA loaded.


CUDA toolchain: 


- runtime 13.0, artifact installation
- driver 580.119.2 for 13.1
- compiler 13.1

CUDA libraries: 
- CUBLAS: 13.1.0
- CURAND: 10

.4.0
- CUFFT: 12.0.0
- CUSOLVER: 12.0.4
- CUSPARSE: 

12.6.3
- CUPTI: 2025.3.1 (API 13.0.1)
- NVML: 13.0.0+580.119.2

Julia packages: 
- CUDA: 5.9.6
- GPUArrays: 11.3.4
- GPUCompiler: 1.8.2
- KernelAbstractions: 0.9.39
- CUDA_Driver_jll: 13.1.0+2
- CUDA_Compiler_jll: 0.4.1+1
- CUDA_Runtime_jll: 0.19.2+0

Toolchain:
- Julia: 1.12.4
- LLVM: 18.1.7

1 device:


  0: NVIDIA RTX 4500 Ada Generation (sm_89, 23.506 GiB / 23.994 GiB available)
CUDA functional = true
CUDA has GPU    = true
Octopus loaded.


In [2]:
octopus_load_error === nothing || error("Octopus did not load; inspect the first cell output.")

function clone_beam_cpu(beam)
    rep = beam.rep
    rep2 = Phase6DRep(copy(rep.x), copy(rep.px), copy(rep.y), copy(rep.py), copy(rep.z), copy(rep.pz))
    return Beam{CPUThreadsBackend, typeof(beam.params), typeof(rep2)}(beam.params, rep2)
end

function clone_beam_gpu(beam)
    rep = beam.rep
    grep = Phase6DRep(
        CUDA.CuArray(rep.x), CUDA.CuArray(rep.px), CUDA.CuArray(rep.y),
        CUDA.CuArray(rep.py), CUDA.CuArray(rep.z), CUDA.CuArray(rep.pz),
    )
    return Beam{CUDABackend, typeof(beam.params), typeof(grep)}(beam.params, grep)
end

function beam_pair(N; seed1=1, seed2=2)
    p = CPUThreadsExecutionPolicy()
    b1 = Beam(N, p, Float64;
        sigma=(1e-4, 1e-5, 1e-2), alpha=(0.0, 0.0), rng=Random.MersenneTwister(seed1),
        charge=1.0, mc2=PMASS_EV, E0=275e9, r0=RE * ME0 / PMASS_EV, npart=1e10,
    )
    b2 = Beam(N, p, Float64;
        sigma=(1e-4, 1e-5, 1e-2), alpha=(0.0, 0.0), rng=Random.MersenneTwister(seed2),
        charge=-1.0, mc2=EMASS_EV, E0=10e9, r0=RE, npart=1e10,
    )
    return b1, b2
end

function max_coordinate_error(cpu_beam, gpu_beam)
    c = coordinate_arrays(cpu_beam.rep)
    g = map(Array, coordinate_arrays(gpu_beam.rep))
    return maximum(maximum(abs.(c[i] .- g[i])) for i in 1:6)
end

function relative_coordinate_error(cpu_beam, gpu_beam)
    c = coordinate_arrays(cpu_beam.rep)
    g = map(Array, coordinate_arrays(gpu_beam.rep))
    num = maximum(maximum(abs.(c[i] .- g[i])) for i in 1:6)
    den = maximum(maximum(abs.(c[i])) for i in 1:6)
    return num / max(den, eps(Float64))
end

relative_coordinate_error (generic function with 1 method)

In [3]:
octopus_load_error === nothing || error("Octopus did not load; inspect the first cell output.")

function slicing_config(method)
    if method == :specified
        return LongitudinalSlicing(method=:specified, positions=[-1.0, 0.0, 1.0])
    else
        return LongitudinalSlicing(nslices=5, method=method, resolution=50)
    end
end

function run_collision_case(method; N=20_000, warmup=true,
                            virtual_drift=:hirata,
                            include_sigma_xy=false,
                            longitudinal_kick=true,
                            batch_mode=:wavefront)
    solver = GaussianPoissonSolver(;
        slicing=slicing_config(method),
        virtual_drift=virtual_drift,
        include_sigma_xy=include_sigma_xy,
        longitudinal_kick=longitudinal_kick,
        batch_mode=batch_mode,
    )

    if warmup
        warm_N = min(N, 2_000)
        wb1, wb2 = beam_pair(warm_N; seed1=101, seed2=202)
        wb1_cpu = clone_beam_cpu(wb1)
        wb2_cpu = clone_beam_cpu(wb2)
        wb1_gpu = clone_beam_gpu(wb1)
        wb2_gpu = clone_beam_gpu(wb2)
        collide!(solver, wb1_cpu, wb2_cpu, CPUThreadsBackend)
        collide!(solver, wb1_gpu, wb2_gpu, CUDABackend)
        CUDA.synchronize()
    end

    b1_base, b2_base = beam_pair(N)
    b1_cpu = clone_beam_cpu(b1_base)
    b2_cpu = clone_beam_cpu(b2_base)
    b1_gpu = clone_beam_gpu(b1_base)
    b2_gpu = clone_beam_gpu(b2_base)

    cpu_time = @elapsed cpu_lum = collide!(solver, b1_cpu, b2_cpu, CPUThreadsBackend)
    gpu_time = @elapsed begin
        gpu_lum = collide!(solver, b1_gpu, b2_gpu, CUDABackend)
        CUDA.synchronize()
    end

    return (
        method=method,
        N=N,
        cpu_lum=cpu_lum,
        gpu_lum=gpu_lum,
        abs_lum_error=abs(cpu_lum - gpu_lum),
        rel_lum_error=abs(cpu_lum - gpu_lum) / max(abs(cpu_lum), eps(Float64)),
        max_coord_error=max(max_coordinate_error(b1_cpu, b1_gpu), max_coordinate_error(b2_cpu, b2_gpu)),
        rel_coord_error=max(relative_coordinate_error(b1_cpu, b1_gpu), relative_coordinate_error(b2_cpu, b2_gpu)),
        cpu_time=cpu_time,
        gpu_time=gpu_time,
        speedup=cpu_time / gpu_time,
    )
end

run_collision_case (generic function with 1 method)

In [4]:
methods = (:equal_width, :equal_area, :specified, :equal_count)

if octopus_load_error !== nothing
    println("Skipping test because Octopus did not load; inspect the first cell output.")
elseif !cuda_ok
    println("Skipping CUDA strong-strong collision test because this kernel cannot see a CUDA GPU.")
else
    results = [run_collision_case(method; N=20_000) for method in methods]
    for r in results
        println(r)
    end
end

(method

 = :equal_width, N = 20000, cpu_lum = 7.954781596935949e27, gpu_lum = 7.954781596935946e27, abs_lum_error = 3.298534883328e12, rel_lum_error = 4.146606469495707e-16, max_coord_error = 1.90074193363865e-18, rel_coord_error = 4.653731756565411e-17, cpu_time = 0.051224075, gpu_time = 18.821353433, speedup = 0.0027215935975245763)
(method = :equal_area, N = 20000, cpu_lum = 7.956850745618754e27, gpu_lum = 7.956850745618751e27, abs_lum_error = 2.199023255552e12, rel_lum_error = 2.763685440201123e-16, max_coord_error = 1.8516140226979005e-18, rel_coord_error = 4.5334481371889433e-17, cpu_time = 0.088262125, gpu_time = 0.041686152, speedup = 2.117300848492804)
(method = :specified, N = 20000, cpu_lum = 7.956501244362416e27, gpu_lum = 7.956501244362415e27, abs_lum_error = 1.099511627776e12, rel_lum_error = 1.3819034196155749e-16, max_coord_error = 1.836367429647323e-18, rel_coord_error = 4.4961187380721085e-17, cpu_time = 0.040968794, gpu_time = 0.220141819, speedup = 0.18610182375207868)
(met

## Larger Timing Run

Run this cell after the small validation succeeds. Increase `N` cautiously because strong-strong collision does sequential slice-pair processing.

In [5]:
if octopus_load_error === nothing && cuda_ok
    large = run_collision_case(:equal_width; N=200_000)
    println(large)
end

(method

 = :equal_width, N = 200000, cpu_lum = 7.972446904787709e27, gpu_lum = 7.972446904787702e27, abs_lum_error = 6.597069766656e12, rel_lum_error = 8.274836879370435e-16, max_coord_error = 1.9668105035244854e-18, rel_coord_error = 4.154698920276515e-17, cpu_time = 0.873572423, gpu_time = 0.156467084, speedup = 5.583106687154724)


## Full Example Command

The runnable example can also be launched from a notebook cell once CUDA is visible:

```julia
ENV["OCTOPUS_USE_GPU"] = "1"
include(joinpath(@__DIR__, "..", "examples", "strong_strong_tracking.jl"))
```

In [6]:
println("Julia executable: ", Base.julia_cmd().exec[1])
println("Julia version: ", VERSION)
println("Project: ", Base.active_project())
println("Threads: ", Threads.nthreads())
println("DEPOT_PATH:")
foreach(println, DEPOT_PATH)
println("LOAD_PATH:")
foreach(println, LOAD_PATH)

Julia executable: /cfs/ad/dxu/packages/julias/julia-1.12.4/bin/julia
Julia version: 

1.12.4
Project: /cfs/ad/dxu/Library/Julia/Octopus/Project.toml
Threads: 1
DEPOT_PATH:
/home/cfsd/dxu/.julia
/cfs/ad/dxu/packages/julias/julia-1.12.4/local/share/julia
/cfs/ad/dxu/packages/julias/julia-1.12.4/share/julia
LOAD_PATH:
@
@v#.#
@stdlib
